# Migración Clínica X → Flowww

## Tabla clients

El saldo lo guardo en `ClientCustom1`.

Mantengo también a los empleados, porque además son clientes de la clínica y tienen tratamientos, presupuestos, etc.

El `ClinicID` se toma del archivo de origen (`df_origen["ClinicID"]`). Si ese campo es nulo, está vacío o vale 0, se le asigna el valor por defecto 1 mediante la función `aplicar_default_clinicid`.

In [ ]:
import pandas as pd
import pyodbc
import os
import re
from decimal import Decimal, InvalidOperation

# =========================================================
# CONFIG
# =========================================================
DB_NAME = "CLINICA_ORIGEN"

RUTA_PLANTILLA = r"C:\Migracion\ClinicaX\Origen\Clients\clients.csv"
RUTA_SALIDA    = r"C:\Migracion\ClinicaX\Destino\clients.csv"

CLINICID_DEFAULT = "1"

# =========================================================
# UTILIDADES
# =========================================================
def convertir_fecha(valor) -> str:
    if valor is None or pd.isna(valor):
        return ""
    try:
        return pd.to_datetime(valor).strftime("%d/%m/%Y")
    except Exception:
        return ""

def limpiar_entero_como_texto(x) -> str:
    if x is None or pd.isna(x):
        return ""
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s

def limpiar_telefono(v) -> str:
    if v is None or pd.isna(v):
        return ""

    if isinstance(v, (int, float)):
        try:
            return str(int(v))
        except Exception:
            return ""

    s = str(v).strip()

    if "E+" in s.upper() or "E-" in s.upper():
        try:
            s_norm = s.replace(",", ".")
            return str(int(Decimal(s_norm)))
        except (InvalidOperation, ValueError):
            return ""

    numeros = re.findall(r"\d+", s)
    return "".join(numeros)

def normalizar_sexo(v) -> str:
    if v is None or pd.isna(v):
        return "F"

    s = str(v).strip().upper()
    if s in ["", "NULL", "NONE", "N/A"]:
        return "F"

    if s in ["M", "F"]:
        return s

    if s in ["H", "HOMBRE"]:
        return "M"

    if s in ["V", "MUJER"]:
        return "F"

    return "F"

def aplicar_default_clinicid(v, default_value: str) -> str:
    if v is None or pd.isna(v):
        return default_value

    if isinstance(v, (int, float)):
        try:
            return default_value if float(v) == 0 else str(int(v))
        except Exception:
            return default_value

    s = str(v).strip()
    if s == "":
        return default_value

    su = s.upper()
    if su in ["NULL", "NONE", "N/A"]:
        return default_value

    s_clean = s[:-2] if s.endswith(".0") else s
    if s_clean == "0":
        return default_value

    return s_clean

def normalizar_cp(v) -> str:
    """
    CP como texto de 5 dígitos.
    - Quita .0 si viene tipo 8028.0
    - Extrae dígitos
    - Si quedan 4 dígitos, rellena con 0 delante
    """
    if v is None or pd.isna(v):
        return ""

    s = str(v).strip()
    if s.endswith(".0"):
        s = s[:-2]

    digits = "".join(re.findall(r"\d+", s))
    if digits == "":
        return ""

    # Si tiene más de 5, dejamos los últimos 5 (por si viene con prefijos raros)
    if len(digits) > 5:
        digits = digits[-5:]

    # Si tiene menos de 5, padding a la izquierda
    if len(digits) < 5:
        digits = digits.zfill(5)

    return digits

# =========================================================
# 1) CONEXIÓN SQL SERVER
# =========================================================
print("Conectando a SQL Server...")

conn = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# =========================================================
# 2) ORIGEN: VISTA SQL dbo.clients
# =========================================================
df_origen = pd.read_sql("SELECT * FROM dbo.clients;", conn)

print("Columnas origen:", len(df_origen.columns))
print("Filas encontradas en origen:", len(df_origen))

# =========================================================
# 3) PLANTILLA DESTINO (FLOWWW)
# =========================================================
df_plantilla = pd.read_csv(RUTA_PLANTILLA, sep=";", encoding="utf-8-sig")
columnas_destino = df_plantilla.columns.tolist()

print("Columnas destino:", len(columnas_destino))

# =========================================================
# 4) MAPEO ORIGEN → DESTINO
#    ✅ saldoclinica -> ClientCustom1 (NO CustomClient1)
# =========================================================
mapeo = {
    "ClientID": "personaID",
    "ClinicID": "codclinica",
    "ClientName": "nombre",
    "ClientSurname1": "apellido1",
    "ClientSurname2": "apellido2",
    "ClientSex": "sexo",
    "ClientNIF": "dnicif",
    "ClientEmail": "email",
    "ClientBirthDate": "fechanacimiento",
    "ClientDate": "fechaalta",
    "ClientPhone1": "telefonofijo",
    "ClientPhone2": "movil",
    "ClientPhone3": "fax",
    "ClientAddress": "direccion",
    "ClientPostCosde": "codigopostal",
    "ClientCity": "poblacion",
    "ClientProvince": "provincia",

    # ✅ AQUÍ EL CAMBIO CLAVE:
    "ClientCustom1": "saldoclinica",
}

# =========================================================
# 5) CONSTRUIR DATAFRAME FINAL
# =========================================================
df_final = pd.DataFrame(index=df_origen.index)

for col_dest in columnas_destino:
    if col_dest == "ClientGeneralConditions":
        df_final[col_dest] = "0"
        continue

    col_origen = mapeo.get(col_dest)

    if col_origen and col_origen in df_origen.columns:
        df_final[col_dest] = df_origen[col_origen]
    else:
        df_final[col_dest] = ""

# =========================================================
# 6) FORMATO FECHAS
# =========================================================
for col_fecha in ["ClientBirthDate", "ClientDate"]:
    if col_fecha in df_final.columns:
        df_final[col_fecha] = df_final[col_fecha].apply(convertir_fecha)

# =========================================================
# 7) LIMPIEZA TELÉFONOS
# =========================================================
for col in ["ClientPhone1", "ClientPhone2", "ClientPhone3"]:
    if col in df_final.columns:
        df_final[col] = df_final[col].apply(limpiar_telefono)

# =========================================================
# 8) NORMALIZAR SEXO
# =========================================================
if "ClientSex" in df_final.columns:
    df_final["ClientSex"] = df_final["ClientSex"].apply(normalizar_sexo)

# =========================================================
# 9) NORMALIZAR CP (para no perder ceros)
# =========================================================
if "ClientPostCosde" in df_final.columns:
    df_final["ClientPostCosde"] = df_final["ClientPostCosde"].apply(normalizar_cp)

# =========================================================
# 10) LIMPIEZA NUMÉRICOS (sin tocar CP)
# =========================================================
for col in ["ClientID", "ClinicID"]:
    if col in df_final.columns:
        df_final[col] = df_final[col].apply(limpiar_entero_como_texto)

# =========================================================
# 11) DEFAULT ClinicID (vacío/null/0 -> 1)
# =========================================================
if "ClinicID" in df_final.columns:
    df_final["ClinicID"] = df_final["ClinicID"].apply(lambda x: aplicar_default_clinicid(x, CLINICID_DEFAULT))

# Orden exacto según plantilla
df_final = df_final[columnas_destino]

# =========================================================
# 12) GUARDAR CSV FINAL
# =========================================================
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)

df_final.to_csv(
    RUTA_SALIDA,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ CSV generado correctamente:")
print(RUTA_SALIDA)
print(f"ℹ️ Regla aplicada: ClinicID vacío/null/0 => {CLINICID_DEFAULT}")
print("ℹ️ Mapeo aplicado: saldoclinica -> ClientCustom1")
print("ℹ️ CP normalizado a 5 dígitos (ej: 08028)")


## Tabla diary_gen (agenda futura)

Requiere subir antes el mapeo de doctores.

In [ ]:
import os
import datetime
import pandas as pd
import pyodbc

# =========================================================
# CONFIG
# =========================================================
DB_NAME = "CLINICA_ORIGEN"

RUTA_SALIDA = r"C:\Migracion\ClinicaX\Destino\diarygen.csv"

# =========================================================
# UTILIDADES
# =========================================================
def convertir_fecha(v):
    if v is None or pd.isna(v):
        return ""
    try:
        return pd.to_datetime(v).strftime("%d/%m/%Y")
    except Exception:
        return ""

def convertir_hora(v):
    if v is None or pd.isna(v):
        return ""
    try:
        return pd.to_datetime(v).strftime("%H:%M")
    except Exception:
        return ""

# =========================================================
# 1) CONEXIÓN SQL SERVER
# =========================================================
print("Conectando a SQL Server...")

conn = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# =========================================================
# 2) QUERY SQL (YA CON MAPEOS HECHOS)
# =========================================================
query = """
SELECT
    a.codclinica,
    a.codgabinete,
    a.fecha,
    a.horacita,
    a.horafin,
    a.fechacreacion,
    a.anotaciones,

    cli.personaID  AS ClientID,
    prof.personaID AS DiaryGUserID

FROM agenda AS a
LEFT JOIN person_dup AS cli
    ON cli.codpersona = a.codpersona
   AND cli.codempresa = a.codempresa
LEFT JOIN person_dup AS prof
    ON prof.codpersona = a.codprofesional
   AND prof.codempresa = a.codempresa

WHERE a.codpersona <> '00000000-0000-0000-0000-000000000000'
    AND a.fecha >= DATEADD(DAY, 1, CAST(GETDATE() AS DATE));
"""

df = pd.read_sql(query, conn)

print(f"Filas recuperadas desde SQL: {len(df)}")

# =========================================================
# 3) CONSTRUIR DATAFRAME FINAL (FLOWW)
# =========================================================
df_final = pd.DataFrame()

# ✅ NUEVAS COLUMNAS: autoincrement desde 1 (mismo valor en ambas)
df_final["DiaryGID"] = range(1, len(df) + 1)
df_final["DiaryID"]  = df_final["DiaryGID"]

# Identificadores
df_final["DiaryGClientID"] = df["ClientID"]

# Usuarios
df_final["DiaryGSUserID"] = 0  # fijo por defecto
df_final["DiaryGUserID"] = df["DiaryGUserID"].fillna(0).astype(int)

# Clínica / cabina
df_final["DiaryGClinicID"] = 1  # fijo por defecto
df_final["DiaryGCabinetNum"] = 1  # fijo por defecto

# Fecha y horas
df_final["DiaryGDate"] = df["fecha"].apply(convertir_fecha)
df_final["DiaryGStart"] = df["horacita"].apply(convertir_hora)
df_final["DiaryGEnd"] = df["horafin"].apply(convertir_hora)

# Metadatos
df_final["DiaryGCreated"] = df["fechacreacion"].apply(convertir_fecha)
df_final["DiaryGComments"] = df["anotaciones"]

# Tratamiento (vacío por ahora)
df_final["DiaryZoneID"] = ""

# =========================================================
# 4) GUARDAR CSV
# =========================================================
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)

df_final.to_csv(
    RUTA_SALIDA,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("✅ CSV generado correctamente:")
print(RUTA_SALIDA)


## Tabla budgets_gen (presupuestos)

Requiere hacer antes el mapeo de tratamientos a servicios y productos.

`BudgetPrice` = `importebi` (importe sin IVA), de la vista SQL `Budget_gen`.

In [ ]:
import os
import pandas as pd
import pyodbc

# =========================================================
# CONFIG
# =========================================================
DB_NAME = "CLINICA_ORIGEN"
RUTA_SALIDA = r"C:\Migracion\ClinicaX\Destino\budgets_gen.csv"

COLUMNAS_FINALES = [
    "BudgetGStatus",
    "BudgetGID",
    "BudgetID",
    "BudgetGClinicID",
    "BudgetGClientID",
    "BudgetGUserID",
    "BudgetGDoctorID",
    "BudgetGDate",
    "BudgetGValidity",
    "BudgetProductID",
    "BudgetUnits",
    "BudgetPrice",
    "BudgetDiscountAmount",
    "BudgetVAT",
    "BudgetComments",
]

# =========================================================
# 1) CONEXIÓN SQL SERVER
# =========================================================
print("🔌 Conectando a SQL Server...")

conn = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# =========================================================
# 2) LEER VISTA ORIGEN
# =========================================================
query = "SELECT * FROM budgets_gen;"
df = pd.read_sql(query, conn)

print(f"📊 Filas recuperadas: {len(df)}")
print(f"📊 Columnas origen: {list(df.columns)}")

# =========================================================
# 3) UTILIDADES
# =========================================================
def convertir_fecha(v):
    if v is None or pd.isna(v):
        return ""
    try:
        return pd.to_datetime(v).strftime("%d/%m/%Y")
    except Exception:
        return ""

def limpiar_entero_como_texto(x):
    if x is None or pd.isna(x):
        return ""
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    try:
        return str(int(float(s)))
    except Exception:
        return ""

# =========================================================
# 4) NORMALIZACIÓN / FORMATO
# =========================================================

# IDs y unidades a texto entero limpio
for col in ["BudgetGID", "BudgetID", "BudgetClientID", "BudgetDoctorID", "BudgetUnits"]:
    if col in df.columns:
        df[col] = df[col].apply(limpiar_entero_como_texto)

# Producto
if "BudgetProductID" in df.columns:
    df["BudgetProductID"] = (
        pd.to_numeric(df["BudgetProductID"], errors="coerce")
        .fillna(2060)
        .astype(int)
    )

# Importes (precio + descuento)
for col in ["BudgetPrice", "BudgetDiscount"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).round(2)

# Fecha BudgetCreateDate -> BudgetGDate
if "BudgetCreateDate" in df.columns:
    df["BudgetGDate"] = df["BudgetCreateDate"].apply(convertir_fecha)
else:
    df["BudgetGDate"] = ""

# =========================================================
# 5) VALORES FIJOS FLOWWW
# =========================================================
df["BudgetGClinicID"] = 1
df["BudgetGUserID"] = 1
df["BudgetVAT"] = 0.21
df["BudgetGValidity"] = ""

# =========================================================
# 6) RENOMBRES A COLUMNAS FINALES
# =========================================================
df = df.rename(columns={
    "BudgetClientID": "BudgetGClientID",
    "BudgetDoctorID": "BudgetGDoctorID",
    "BudgetDiscount": "BudgetDiscountAmount",
    # BudgetGStatus viene ya desde SQL
})

# =========================================================
# 7) ASEGURAR COLUMNAS FINALES (si falta alguna, crearla vacía)
# =========================================================
for col in COLUMNAS_FINALES:
    if col not in df.columns:
        df[col] = ""

# =========================================================
# 8) DEJAR SOLO LAS COLUMNAS PERMITIDAS
# =========================================================
df_final = df[COLUMNAS_FINALES].copy()

# =========================================================
# 9) GUARDAR CSV FINAL
# =========================================================
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)

df_final.to_csv(
    RUTA_SALIDA,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("✅ CSV 'budgets_gen' generado correctamente")
print("📌 Columnas finales:", list(df_final.columns))
print(RUTA_SALIDA)


## Tabla clients_crm (historia clínica)

In [ ]:
import os
import pandas as pd
import pyodbc

# =========================================================
# CONFIG
# =========================================================
DB_NAME = "CLINICA_ORIGEN"
RUTA_SALIDA = r"C:\Migracion\ClinicaX\Destino\Clients_crm.csv"  # Ruta de salida del CSV

# =========================================================
# CONEXIÓN SQL SERVER
# =========================================================
print("🔌 Conectando a SQL Server...")

conn = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"  # Asegúrate de que tu servidor esté correcto
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# =========================================================
# 2) LEER DATOS DE HISTORIA CLÍNICA Y CLIENTES
# =========================================================
query = """
SELECT 
    p.personaID,
    p.nombre,
    p.apellido1,
    p.apellido2,
    h.*
FROM historiaclinica AS h
JOIN clients AS p
    ON h.codpersona = p.codpersona;
"""

df = pd.read_sql(query, conn)

print(f"📊 Filas recuperadas desde SQL: {len(df)}")
print(f"📊 Columnas recuperadas desde SQL: {list(df.columns)}")

# =========================================================
# 3) LIMPIEZA DE DATOS (Formato de fechas)
# =========================================================

# Convertir las fechas a un formato consistente
def convertir_fecha(v):
    if v is None or pd.isna(v):
        return ""
    try:
        return pd.to_datetime(v).strftime("%d/%m/%Y")
    except Exception:
        return ""

# Limpiar las fechas en las columnas correspondientes
for col_fecha in ["fechacreacion"]:  # Aquí puedes añadir más columnas de fecha si es necesario
    if col_fecha in df.columns:
        df[col_fecha] = df[col_fecha].apply(convertir_fecha)

# =========================================================
# 4) MODIFICACIÓN DE DATOS
# =========================================================
# Asignar valores por defecto según lo solicitado
df["ClientCID"] = range(1, len(df) + 1)  # Autoincrementar desde 1
df["ClinicID"] = 1  # Asignar 1 a todas las filas por defecto

# Asignar ClientID desde personaID
df["ClientID"] = df["personaID"]

# Asignar ClientCComments desde 'anotaciones'
df["ClientCComments"] = df["anotaciones"]

# Asignar ClientCDate desde 'fechacreacion'
df["ClientCDate"] = df["fechacreacion"]

# Limpiar las columnas que no son necesarias
df = df[['ClientCID', 'ClinicID', 'ClientID', 'ClientCComments', 'ClientCDate']]

# =========================================================
# 5) GUARDAR CSV FINAL
# =========================================================
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)

df.to_csv(
    RUTA_SALIDA,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("✅ CSV 'Clients_crm' generado correctamente:")
print(RUTA_SALIDA)


## Añadir los registros de la tabla `trmient` a clients_crm

In [ ]:
import os
import csv
import logging
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# =========================================================
# LOGGING (solo mensaje)
# =========================================================
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(message)s"))
logger.handlers = [handler]
logger.propagate = False

# =========================================================
# RUTAS (ÚNICA FUENTE DE VERDAD)
# =========================================================
DESTINO_DIR = r"C:\Migracion\ClinicaX\Destino"
OUTPUT_FILE = os.path.join(DESTINO_DIR, "Clients_crm.csv")

EXPECTED_COLS = [
    "ClientCID", "ClinicID", "ClientID",
    "ClientCComments", "ClientCDate"
]

# =========================================================
# ENV
# =========================================================
load_dotenv(".env")

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

if not DB_SERVER or not DB_NAME:
    raise ValueError("Faltan variables DB_SERVER / DB_NAME")

# =========================================================
# CONEXIÓN SQL
# =========================================================
logger.info("🔌 Conectando a SQL Server con SQLAlchemy (Windows Auth)...")

odbc_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server={DB_SERVER};"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
    fast_executemany=True
)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

logger.info("✅ Conexión OK")

# =========================================================
# 1) LEER Client_crm.csv DESDE DESTINO
# =========================================================
if os.path.exists(OUTPUT_FILE):
    df_origen = pd.read_csv(
        OUTPUT_FILE,
        sep=";",
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    df_origen.columns = [c.strip().replace("\ufeff", "") for c in df_origen.columns]
    logger.info(f"📂 Clients_crm.csv leído desde Destino")
else:
    logger.warning("⚠️ Client_crm.csv no existe. Se creará uno nuevo.")
    df_origen = pd.DataFrame(columns=EXPECTED_COLS)

filas_antes = len(df_origen)

# =========================================================
# 2) ÚLTIMO ClientCID (SEGURO)
# =========================================================
if "ClientCID" in df_origen.columns and not df_origen.empty:
    last_clientcid = (
        pd.to_numeric(df_origen["ClientCID"], errors="coerce")
        .dropna()
        .astype(int)
        .max()
    )
    last_clientcid = int(last_clientcid) if pd.notna(last_clientcid) else 0
else:
    last_clientcid = 0

# =========================================================
# 3) QUERY SQL
# =========================================================
query = """
SELECT 
p.personaID,
p.nombre,
p.apellido1,
p.apellido2,
 t.*
from trmient as t
Join person_dup as p
on t.codpersona = p.codpersona;
"""

df_sql = pd.read_sql(query, engine)

# =========================================================
# 4) TRANSFORMAR NUEVOS REGISTROS
# =========================================================
def clean_date(v):
    d = pd.to_datetime(v, errors="coerce")
    return "" if pd.isna(d) else d.strftime("%d/%m/%Y")

df_sql = df_sql.rename(columns={
    "personaID": "ClientID",
    "fechacreacion": "ClientCDate"
})

df_sql["ClinicID"] = "1"
df_sql["ClientCID"] = range(last_clientcid + 1, last_clientcid + 1 + len(df_sql))
df_sql["ClientCDate"] = df_sql["ClientCDate"].apply(clean_date)

df_sql["ClientID"] = (
    pd.to_numeric(df_sql["ClientID"], errors="coerce")
    .fillna(0)
    .astype(int)
    .astype(str)
)

df_sql["ClientCID"] = df_sql["ClientCID"].astype(str)

# Concatenación personalizada para ClientCComments con las columnas correctas
df_sql["ClientCComments"] = df_sql.apply(
    lambda row: f"{row['anotacionestratamedicats']}\n"  # Texto de anotacionestratamedicats
                f"anotaciones:\n"  # Texto fijo "anotaciones:"
                f"{row['anotaciones']}",  # Texto de anotaciones
    axis=1
)

df_sql["ClientCComments"] = df_sql["ClientCComments"].fillna("").astype(str)

df_sql = df_sql[EXPECTED_COLS]

# =========================================================
# 5) CONCATENAR + GUARDAR EN DESTINO
# =========================================================
df_final = pd.concat([df_origen, df_sql], ignore_index=True)

os.makedirs(DESTINO_DIR, exist_ok=True)

df_final.to_csv(
    OUTPUT_FILE,
    index=False,
    sep=";",
    encoding="utf-8-sig",
    quoting=csv.QUOTE_MINIMAL
)

# =========================================================
# 6) OUTPUT FINAL
# =========================================================
logger.info(f"📄 Filas antes: {filas_antes}")
logger.info(f"➕ Registros añadidos: {len(df_sql)}")
logger.info(f"📊 Total final: {len(df_final)}")
logger.info("=" * 60)


## Tabla laser_gen (historial de tratamientos): join con `fichamedica`, sin `codigotrat`

Uso SQLAlchemy y variables de entorno.

In [ ]:
import os
import logging
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# =========================================================
# 0) CARGAR .env
# =========================================================
load_dotenv(".env")

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

if not DB_SERVER or not DB_NAME:
    raise ValueError("Faltan DB_SERVER o DB_NAME en el .env")

# =========================================================
# 1) CONEXIÓN SQL SERVER CON SQLALCHEMY (Windows Auth)
# =========================================================
logger.info("🔌 Conectando a SQL Server con SQLAlchemy (Windows Auth)...")

odbc_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server={DB_SERVER};"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
    fast_executemany=True,
)

# Test de conexión
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
logger.info("✅ Conexión OK")

# =========================================================
# 2) QUERY 
# =========================================================
query = """
SELECT 
    p.personaID,
    p.nombre,
    p.apellido1,
    p.apellido2,
    f.*
FROM fichamedica AS f
JOIN person_dup AS p
    ON f.codpersona = p.codpersona;
"""

df = pd.read_sql(query, engine)
logger.info(f"📊 Filas: {len(df)}")
logger.info(f"📊 Columnas: {df.columns.tolist()}")

# =========================================================
# 3) TRANSFORMACIONES (mapeo fechacreacion -> LaserGDate)
# =========================================================
def convertir_fecha(valor) -> str:
    if valor is None or pd.isna(valor):
        return ""
    try:
        return pd.to_datetime(valor).strftime("%d/%m/%Y")
    except Exception:
        return ""

# Comentarios
df["LaserGComments"] = df["titulo"] if "titulo" in df.columns else ""

# Fecha
if "fechacreacion" in df.columns:
    df["LaserGDate"] = df["fechacreacion"].apply(convertir_fecha)
else:
    logger.warning("⚠️ No existe 'fechacreacion'. LaserGDate irá vacío.")
    df["LaserGDate"] = ""

# =========================================================
# 3.1) COLUMNAS SEGÚN CSV VÁLIDO (NOMBRES EXACTOS)
# =========================================================
df["LaserGID"] = range(1, len(df) + 1)
df["LaserID"] = df["LaserGID"]

df["LaserGClinicID"] = 1
df["LaserGCabinetNum"] = 1
df["LaserGClientID"] = df["personaID"]
df["LaserGUserID"] = 0

df["LaserGStart"] = "00:00:00"

# Columnas requeridas adicionales
df["LaserZoneID"] = 0
df["LaserSessionID"] = 0
df["DentalID"] = 0
df["DentalPieceStart"] = 0
df["DentalPieceEnd"] = 0

# =========================================================
# 3.2) DATAFRAME FINAL (MISMO ORDEN QUE CSV SUBIDO)
# =========================================================
df_final = df[
    [
        "LaserGID",
        "LaserID",
        "LaserGClinicID",
        "LaserGCabinetNum",
        "LaserGClientID",
        "LaserGUserID",
        "LaserGDate",
        "LaserGStart",
        "LaserZoneID",
        "LaserSessionID",
        "LaserGComments",
        "DentalID",
        "DentalPieceStart",
        "DentalPieceEnd",
    ]
]

# =========================================================
# 4) EXPORT CSV
# =========================================================
RUTA_SALIDA = r"C:\Migracion\ClinicaX\Destino\Laser_gen.csv"
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)

df_final.to_csv(RUTA_SALIDA, sep=";", index=False, encoding="utf-8-sig")
logger.info(f"✅ CSV generado: {RUTA_SALIDA}")


## laser_gen con `codigotrat`: join con `detallealbaranhistorico`

In [ ]:
import os
import logging
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# =========================================================
# 0) CARGAR .env
# =========================================================
load_dotenv(".env")

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

if not DB_SERVER or not DB_NAME:
    raise ValueError("Faltan DB_SERVER o DB_NAME en el .env")

# =========================================================
# 1) CONEXIÓN SQL SERVER CON SQLALCHEMY (Windows Auth)
# =========================================================
logger.info("🔌 Conectando a SQL Server con SQLAlchemy (Windows Auth)...")

odbc_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server={DB_SERVER};"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
    fast_executemany=True,
)

# Test de conexión
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
logger.info("✅ Conexión OK")

# =========================================================
# 2) QUERY (NUEVA)
# =========================================================
query = """
SELECT 
    p.personaID,
    p.nombre,
    p.apellido1,
    p.apellido2,
    map.codigotrat,
    map.ProductID AS LaserZoneID,
    deta.*
FROM detallealbaran AS deta
JOIN person_dup AS p
    ON deta.codpersona = p.codpersona
LEFT JOIN product_map AS map
    ON deta.codigotrat = map.codigotrat;
"""

df = pd.read_sql(query, engine)
logger.info(f"📊 Filas: {len(df)}")
logger.info(f"📊 Columnas: {df.columns.tolist()}")

# =========================================================
# 3) TRANSFORMACIONES (mapeo fechacreacion -> LaserGDate)
# =========================================================
def convertir_fecha(valor) -> str:
    if valor is None or pd.isna(valor):
        return ""
    try:
        return pd.to_datetime(valor).strftime("%d/%m/%Y")
    except Exception:
        return ""

# Comentarios
df["LaserGComments"] = df["titulo"] if "titulo" in df.columns else ""

# Fecha
if "fechacreacion" in df.columns:
    df["LaserGDate"] = df["fechacreacion"].apply(convertir_fecha)
else:
    logger.warning("⚠️ No existe 'fechacreacion'. LaserGDate irá vacío.")
    df["LaserGDate"] = ""

# =========================================================
# 3.1) COLUMNAS SEGÚN CSV VÁLIDO (NOMBRES EXACTOS)
# =========================================================
df["LaserGID"] = range(1, len(df) + 1)
df["LaserID"] = df["LaserGID"]

df["LaserGClinicID"] = 1
df["LaserGCabinetNum"] = 1
df["LaserGClientID"] = df["personaID"]
df["LaserGUserID"] = 0

df["LaserGStart"] = "00:00:00"

# =========================================================
# LaserZoneID: viene de la query; si viene NULL/NaN/vacío -> 0
# =========================================================
if "LaserZoneID" not in df.columns:
    logger.warning("⚠️ No existe 'LaserZoneID' en la query. Se rellenará con 0.")
    df["LaserZoneID"] = 0
else:
    df["LaserZoneID"] = (
        pd.to_numeric(df["LaserZoneID"], errors="coerce")  # NULL / '' / basura -> NaN
        .fillna(0)                                         # NaN -> 0
        .astype(int)                                       # 217.0 -> 217
    )

# Columnas requeridas adicionales
df["LaserSessionID"] = 0
df["DentalID"] = 0
df["DentalPieceStart"] = 0
df["DentalPieceEnd"] = 0

# =========================================================
# 3.2) DATAFRAME FINAL (MISMO ORDEN QUE CSV SUBIDO)
# =========================================================
df_final = df[
    [
        "LaserGID",
        "LaserID",
        "LaserGClinicID",
        "LaserGCabinetNum",
        "LaserGClientID",
        "LaserGUserID",
        "LaserGDate",
        "LaserGStart",
        "LaserZoneID",
        "LaserSessionID",
        "LaserGComments",
        "DentalID",
        "DentalPieceStart",
        "DentalPieceEnd",
    ]
]

# =========================================================
# 4) EXPORT CSV
# =========================================================
RUTA_SALIDA = r"C:\Migracion\ClinicaX\Destino\Laser_gen.csv"
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)

df_final.to_csv(RUTA_SALIDA, sep=";", index=False, encoding="utf-8-sig")
logger.info(f"✅ CSV generado: {RUTA_SALIDA}")


### clients_crm: generación y actualización del CSV (historia clínica más historial de tratamientos, clients_crm + laser_gen)

Este script genera y mantiene actualizado el fichero `Clients_crm.csv` para la migración de datos a Flowww del entorno de **Clínica X**.
El archivo funciona como **única fuente de verdad**: siempre se lee y se sobrescribe en la misma ubicación.

#### Qué hace
1. **Se conecta a SQL Server** con SQLAlchemy y **autenticación de Windows**.
2. **Lee el archivo `Clients_crm.csv` de la carpeta `Destino`**:
   - Si el archivo existe, lo carga como punto de partida.
   - Si no existe, crea un DataFrame vacío con la estructura que espera Flowww.
3. **Calcula el último `ClientCID`** existente para continuar la numeración sin duplicados.
4. **Ejecuta una consulta SQL** sobre las tablas `fichamedica` y `person_dup` para obtener estos campos:
   - `personaID`
   - `titulo`
   - `observaciones`
   - `fechacreacion`
5. **Limpia y normaliza los textos** (`titulo` y `observaciones`):
   - Decodifica las entidades HTML.
   - Elimina etiquetas HTML, saltos de línea, tabulaciones y caracteres invisibles.
   - Normaliza espacios y separadores.
6. **Construye el campo `ClientCComments`** concatenando:
   - `fichamedica.titulo`
   - `fichamedica.observaciones`
7. **Da a los datos el formato de Flowww**:
   - `ClientID` a partir de `personaID`.
   - `ClientCDate` a partir de `fechacreacion`, con formato `dd/mm/aaaa`.
   - `ClinicID` fijo (`1`).
   - `ClientCID` incremental a partir del último valor existente.
8. **Une los registros nuevos con los existentes** y **sobrescribe** el archivo `Destino/Clients_crm.csv`.

#### Resultado
El script deja actualizado `Clients_crm.csv`: conserva los registros anteriores, añade los nuevos con identificadores consecutivos y mantiene la trazabilidad del origen de los datos en SQL.

In [ ]:
import os
import csv
import re
import html
import logging
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# =========================================================
# LOGGING (solo mensaje)
# =========================================================
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(message)s"))
logger.handlers = [handler]
logger.propagate = False

# =========================================================
# RUTAS (ÚNICA FUENTE DE VERDAD)
# =========================================================
DESTINO_DIR = r"C:\Migracion\ClinicaX\Destino"
OUTPUT_FILE = os.path.join(DESTINO_DIR, "Clients_crm.csv")

EXPECTED_COLS = [
    "ClientCID",
    "ClinicID",
    "ClientID",
    "ClientCComments",
    "ClientCDate"
]

# =========================================================
# ENV
# =========================================================
load_dotenv(".env")

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

if not DB_SERVER or not DB_NAME:
    raise ValueError("Faltan variables DB_SERVER / DB_NAME")

# =========================================================
# CONEXIÓN SQL SERVER
# =========================================================
logger.info("🔌 Conectando a SQL Server con SQLAlchemy (Windows Auth)...")

odbc_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server={DB_SERVER};"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
    fast_executemany=True
)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

logger.info("✅ Conexión OK")

# =========================================================
# 1) LEER Clients_crm.csv DESDE DESTINO
# =========================================================
if os.path.exists(OUTPUT_FILE):
    df_origen = pd.read_csv(
        OUTPUT_FILE,
        sep=";",
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    df_origen.columns = [c.strip().replace("\ufeff", "") for c in df_origen.columns]
    logger.info("📂 Clients_crm.csv leído desde Destino")
else:
    logger.warning("⚠️ Clients_crm.csv no existe. Se creará uno nuevo.")
    df_origen = pd.DataFrame(columns=EXPECTED_COLS)

filas_antes = len(df_origen)

# =========================================================
# 2) ÚLTIMO ClientCID (SEGURO)
# =========================================================
if not df_origen.empty and "ClientCID" in df_origen.columns:
    last_clientcid = (
        pd.to_numeric(df_origen["ClientCID"], errors="coerce")
        .dropna()
        .astype(int)
        .max()
    )
    last_clientcid = int(last_clientcid) if pd.notna(last_clientcid) else 0
else:
    last_clientcid = 0

# =========================================================
# 3) QUERY SQL
# =========================================================
query = """
SELECT 
    p.personaID,
    f.titulo,
    f.observaciones,
    f.fechacreacion
FROM fichamedica AS f
JOIN person_dup AS p
    ON f.codpersona = p.codpersona;
"""

df_sql = pd.read_sql(query, engine)

# =========================================================
# 4) LIMPIEZA FLOWWW (HTML / TEXTOS)
# =========================================================
def clean_text_for_flowww(value: str) -> str:
    if not value:
        return ""

    value = str(value)
    value = html.unescape(value)
    value = re.sub(r"<\s*br\s*/?>", " | ", value, flags=re.IGNORECASE)
    value = re.sub(r"<[^>]+>", "", value)
    value = re.sub(r"[\t\r\n]+", " | ", value)
    value = re.sub(r"[\u200b\u200c\u200d\uFEFF]", "", value)
    value = re.sub(r"\s{2,}", " ", value)
    value = re.sub(r"(\s\|\s)+", " | ", value)

    return value.strip(" |").strip()

def clean_date(v):
    d = pd.to_datetime(v, errors="coerce")
    return "" if pd.isna(d) else d.strftime("%d/%m/%Y")

# =========================================================
# 5) TRANSFORMACIÓN A FORMATO FLOWWW
# =========================================================
df_sql["titulo"] = df_sql["titulo"].fillna("").apply(clean_text_for_flowww)
df_sql["observaciones"] = df_sql["observaciones"].fillna("").apply(clean_text_for_flowww)

df_sql["ClientCComments"] = (
    df_sql["titulo"].str.strip()
    + " - "
    + df_sql["observaciones"].str.strip()
)

df_sql = df_sql.rename(columns={
    "personaID": "ClientID",
    "fechacreacion": "ClientCDate"
})

df_sql["ClinicID"] = "1"
df_sql["ClientCID"] = range(last_clientcid + 1, last_clientcid + 1 + len(df_sql))
df_sql["ClientCDate"] = df_sql["ClientCDate"].apply(clean_date)

df_sql["ClientID"] = (
    pd.to_numeric(df_sql["ClientID"], errors="coerce")
    .fillna(0)
    .astype(int)
    .astype(str)
)

df_sql["ClientCID"] = df_sql["ClientCID"].astype(str)

df_sql = df_sql[EXPECTED_COLS]

# =========================================================
# 6) CONCATENAR + GUARDAR
# =========================================================
df_final = pd.concat([df_origen, df_sql], ignore_index=True)

os.makedirs(DESTINO_DIR, exist_ok=True)

df_final.to_csv(
    OUTPUT_FILE,
    index=False,
    sep=";",
    encoding="utf-8-sig",
    quoting=csv.QUOTE_MINIMAL
)

# =========================================================
# 7) OUTPUT FINAL
# =========================================================
logger.info(f"📄 Filas antes: {filas_antes}")
logger.info(f"➕ Registros añadidos: {len(df_sql)}")
logger.info(f"📊 Total final: {len(df_final)}")
logger.info("=" * 60)


## Tabla files

In [ ]:
import os
import re
import sys
import unicodedata
import pandas as pd

from pathlib import Path
import sys


# ============================================================
# LIBS
# ============================================================


MIGRACION_ROOT = Path(
    os.getenv("LIBS_ROOT", "../..")
)
if str(MIGRACION_ROOT) not in sys.path:
    sys.path.insert(0, str(MIGRACION_ROOT))

from Libs.flowww_utils import (
    leer_archivo_flexible,
    limpiar_texto,
    extraer_dni,
    extract_text_from_pdf,
    extraer_nombre_paciente_pdf,
)

# ============================================================
# CONFIG
# ============================================================
DRY_RUN = False

PDF_FOLDER = r"C:\Migracion\ClinicaX\Export\Pdf"
CLIENTS_CSV = r"C:\Migracion\ClinicaX\Destino\clients.csv"

OUTPUT_DIR = r"C:\Migracion\ClinicaX\Destino"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "files.csv")

AUDIT_DIR = os.path.join(OUTPUT_DIR, "AUDIT_files")
AUDIT_CSV = os.path.join(AUDIT_DIR, "Audit_pdf_no_match.csv")

FILES_LOGICAL_BASE_DIR = "Pdf"

CSV_SEPARATOR = ";"
CSV_ENCODING = "latin1"

# ============================================================
# FILE CLASS
# ============================================================
FILE_CLASS_MAP = {
    1: ["jpg","jpeg","gif","bmp","tif","cdr","psp","psd","wmf","png"],
    2: ["doc","docx","dot","dotx","rtf","wps"],
    3: ["xls","xlsx","csv","xlt","xltx"],
    4: ["mdb","adp","dbf"],
    5: ["txt","asc","log","ini","bat"],
    6: ["zip","ace","rar"],
    7: ["pdf"],
    8: ["htm","html","mht"],
    9: ["mp3","wav","wma","ogg"],
    10:["avi","mpg","ogm"]
}
DEFAULT_CLASS_ID = 99

def get_file_class_id(filename: str) -> int:
    ext = os.path.splitext(filename)[1].lower().replace(".", "")
    for k, v in FILE_CLASS_MAP.items():
        if ext in v:
            return k
    return DEFAULT_CLASS_ID

# ============================================================
# NORMALIZACIÓN
# ============================================================
def normalize_text_upper(s: str) -> str:
    s = limpiar_texto(s)
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^A-Z ]", " ", s.upper())
    return re.sub(r"\s+", " ", s).strip()

# ============================================================
# FECHAS
# ============================================================
def parse_date_from_filename(filename: str) -> str:
    base = os.path.splitext(filename)[0]
    patterns = [
        r"\b(20\d{2})(\d{2})(\d{2})\b",
        r"\b(\d{1,2})[ _\-](\d{1,2})[ _\-](20\d{2})\b",
    ]
    for p in patterns:
        m = re.search(p, base)
        if m:
            g = m.groups()
            if g[0].startswith("20"):
                y, mth, d = g
            else:
                d, mth, y = g
            return f"{d.zfill(2)}/{mth.zfill(2)}/{y}"
    return ""

def extraer_fecha_desde_pdf(texto: str) -> str:
    if not texto:
        return ""
    m = re.search(r"\b(\d{1,2})[/-](\d{1,2})[/-](20\d{2})\b", texto)
    if m:
        return f"{m.group(1).zfill(2)}/{m.group(2).zfill(2)}/{m.group(3)}"
    return ""

# ============================================================
# FILENAME HELPERS (CLAVE)
# ============================================================
def split_compound_names(name: str) -> str:
    """
    MARIAJOSE PEREZ -> MARIA JOSE PEREZ
    """
    if not name:
        return ""
    parts = re.findall(r"[A-ZÀ-Ý]{3,}", name.upper())
    return " ".join(parts) if len(parts) >= 2 else name

# ============================================================
# AUDIT CLASSIFIER
# ============================================================
def classify_audit_pdf(text: str, extracted_name: str) -> tuple[str, str]:
    t = (text or "").strip()
    t_upper = t.upper()
    text_len = len(t)

    if text_len == 0:
        return "no_text_pdf", "empty_text"

    if text_len < 500 and re.search(r"(NOMBRE|PACIENT|PACIENTE|FIRMA)", t_upper):
        return "handwritten_likely", "short_text_form_like"

    if extracted_name:
        tokens = extracted_name.split()
        if len(tokens) <= 2:
            return "name_not_found", "partial_name_only"

    return "name_not_found", "no_name_match"

# ============================================================
# CLIENT MATCHING
# ============================================================
def build_clients_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["ClientName_N"] = df["ClientName"].map(normalize_text_upper)
    df["ClientSurname1_N"] = df["ClientSurname1"].map(normalize_text_upper)
    df["ClientSurname2_N"] = df["ClientSurname2"].map(normalize_text_upper)
    df["FULLNAME"] = (
        df["ClientName_N"] + " " +
        df["ClientSurname1_N"] + " " +
        df["ClientSurname2_N"]
    ).str.strip()
    df["ClientNIF"] = df["ClientNIF"].map(limpiar_texto).str.upper()
    return df

def match_client(clients: pd.DataFrame, name: str, dni: str):
    if dni:
        hit = clients[clients["ClientNIF"] == dni]
        if len(hit) == 1:
            return hit.iloc[0]["ClientID"], "dni"

    if not name:
        return "", "no_name"

    name_norm = normalize_text_upper(name)

    hit = clients[clients["FULLNAME"] == name_norm]
    if len(hit) == 1:
        return hit.iloc[0]["ClientID"], "fullname_exact"

    name_tokens = set(name_norm.split())
    best_score = 0
    best_id = ""

    for _, row in clients.iterrows():
        client_tokens = set(str(row["FULLNAME"]).split())
        score = len(name_tokens & client_tokens)
        if score > best_score:
            best_score = score
            best_id = row["ClientID"]

    if best_score >= 2:
        return best_id, f"token_match_{best_score}"

    return "", "no_match"

# ============================================================
# MAIN
# ============================================================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(AUDIT_DIR, exist_ok=True)

    clients = leer_archivo_flexible(CLIENTS_CSV)
    clients = build_clients_index(clients)

    pdfs = sorted(f for f in os.listdir(PDF_FOLDER) if f.lower().endswith(".pdf"))
    total = len(pdfs)

    matched_rows = []
    audit_rows = []

    file_id = 1

    for pdf in pdfs:
        fname = pdf
        fpath = f"{FILES_LOGICAL_BASE_DIR}/{fname}"

        name_fn_raw, fecha_fn, comments_fn = parse_filename(fname)
        name_fn = split_compound_names(name_fn_raw)

        text = extract_text_from_pdf(os.path.join(PDF_FOLDER, pdf)) or ""

        fecha = fecha_fn or extraer_fecha_desde_pdf(text)
        comments = comments_fn or "DOCUMENTO PDF"

        dni = extraer_dni(text)

        name_pdf = extraer_nombre_paciente_pdf(text)
        client_id, origen = match_client(clients, name_pdf, dni)

        if not client_id and name_fn:
            client_id, origen = match_client(clients, name_fn, "")
            if client_id:
                origen = "filename_match"

        row = {
            "FileID": file_id,
            "FileClientID": client_id,
            "FileName": fname,
            "FileClassID": get_file_class_id(fname),
            "FilePath": fpath,
            "FileDate": fecha,
            "FileComments": comments,
        }

        if client_id:
            matched_rows.append(row)
        else:
            cat, det = classify_audit_pdf(text, name_pdf)
            audit_rows.append({
                **row,
                "MatchResult": origen,
                "AuditCategory": cat,
                "AuditDetail": det
            })

        if file_id % 100 == 0 or file_id == total:
            print(f"📄 Procesados {file_id}/{total}")

        file_id += 1

    df_files = pd.DataFrame(matched_rows)
    df_audit = pd.DataFrame(audit_rows)

    # ============================================================
    # MÉTRICAS
    # ============================================================
    print("\n=== CONTADORES FINALES ===")
    print(f"📁 TOTAL files.csv: {len(df_files)}")
    print(f"📁 TOTAL audit.csv: {len(df_audit)}")
    print(f"📄 TOTAL PDFs procesados: {len(df_files) + len(df_audit)}")

    if not df_audit.empty:
        print("\n=== MÉTRICAS AUDIT ===")
        print(df_audit["AuditCategory"].value_counts())
        print(df_audit["AuditDetail"].value_counts())

    # ============================================================
    # ESCRITURA
    # ============================================================
    if not DRY_RUN:
        os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
        os.makedirs(os.path.dirname(AUDIT_CSV), exist_ok=True)

        df_files.to_csv(OUTPUT_CSV, sep=CSV_SEPARATOR, index=False, encoding=CSV_ENCODING)
        df_audit.to_csv(AUDIT_CSV, sep=";", index=False, encoding="utf-8-sig")

    print("\n=== FIN ===")

if __name__ == "__main__":
    main()


## Files y pictures a partir de la tabla `multimedia` de SQL

Cambia el prefijo según la carpeta de origen real: `PATH_PREFIX = "carpeta"`.

In [ ]:
import os
import sys
import logging
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
from pathlib import Path

# =========================================================
# IMPORT UTILS (EXPLÍCITO, SIN fu., SIN *)
# =========================================================
MIGRACION_ROOT = Path(
    os.getenv("LIBS_ROOT", "../..")
)
if str(MIGRACION_ROOT) not in sys.path:
    sys.path.insert(0, str(MIGRACION_ROOT))

from Libs.flowww_utils import (
    normalizar_timestamp_yyyymmddhhmmss,
    normalizar_fecha_yyyy_mm_dd,
    clean_text_for_flowww_safe,
)

# =========================================================
# LOGGING
# =========================================================
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(message)s"))
logger.handlers = [handler]
logger.propagate = False

# =========================================================
# RUTAS
# =========================================================
BASE_DIR = r"C:\Migracion\ClinicaX"
DESTINO_DIR = os.path.join(BASE_DIR, "Destino")

FILES_OUT = os.path.join(DESTINO_DIR, "files.csv")
PICTURES_OUT = os.path.join(DESTINO_DIR, "pictures.csv")

CSV_SEP = ";"
CSV_ENCODING = "latin1"

DRY_RUN = False  # integrar con tu framework si ya existe

# =========================================================
# PREFIJO PATH (EDITABLE A MANO)
# =========================================================
PATH_PREFIX = "Pdf"
PATH_JOIN = "/"  # si quieres separador, usa "/"

def build_path(nombrearchivo: str) -> str:
    s = "" if nombrearchivo is None else str(nombrearchivo).strip()
    if not s:
        return ""
    s = s.replace("\\", "/")
    prefix = (PATH_PREFIX or "").strip().replace("\\", "/")
    joiner = PATH_JOIN or ""
    if joiner == "/" and prefix.endswith("/"):
        joiner = ""
    return f"{prefix}{joiner}{s}"[:255]

def file_basename(nombrearchivo: str) -> str:
    s = "" if nombrearchivo is None else str(nombrearchivo).strip()
    s = s.replace("\\", "/")
    return os.path.basename(s)[:255]

# =========================================================
# CLASIFICACIÓN POR EXTENSIÓN (REGLA CLÍNICA)
# =========================================================
IMAGE_EXTS = {"jpg", "jpeg", "png", "bmp", "gif", "tif", "tiff", "heic"}

FILE_CLASS_MAP = {
    1: IMAGE_EXTS,
    2: {"doc", "docx", "rtf"},
    3: {"xls", "xlsx", "csv"},
    5: {"txt", "log"},
    6: {"zip", "rar", "7z"},
    7: {"pdf"},
}

def get_file_class_id(filename: str) -> int:
    ext = os.path.splitext(str(filename))[1].lower().replace(".", "")
    for cid, exts in FILE_CLASS_MAP.items():
        if ext in exts:
            return cid
    return 99

# =========================================================
# SQL
# =========================================================
load_dotenv(".env")

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

if not DB_SERVER or not DB_NAME:
    raise ValueError("Faltan variables DB_SERVER / DB_NAME")

logger.info("🔌 Conectando a SQL Server...")

odbc_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server={DB_SERVER};"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
    fast_executemany=True
)

QUERY = """
select
    p.personaID as IDpaciente,
    m.fechacreacion as Fecha,
    m.nombrearchivo
from multimedia as m
left join person_dup as p 
    on p.codpersona = m.codpersona
where personaID is not null
"""

# =========================================================
# MAIN
# =========================================================
def flowww_safe(s):
    return clean_text_for_flowww_safe(
        s,
        ascii_only=True,
        strip_html=False
    )

def main():
    os.makedirs(DESTINO_DIR, exist_ok=True)

    counters = {
        "sql_rows_total": 0,
        "rows_files_csv": 0,
        "rows_pictures_csv": 0,
        "no_match": 0,
        "total_files_csv": 0,
        "total_pictures_csv": 0,
    }

    logger.info("📥 Ejecutando query SQL...")
    df = pd.read_sql(text(QUERY), engine)
    counters["sql_rows_total"] = len(df)

    if df.empty:
        logger.info("⚠️ Query sin resultados")
        logger.info("\n=== CONTADORES ===")
        for k in sorted(counters.keys()):
            logger.info(f"{k}: {counters[k]}")
        return

    df.columns = ["IDpaciente", "Fecha", "nombrearchivo"]

    df["FileName"] = df["nombrearchivo"].map(file_basename)
    df["FlowwwPath"] = df["nombrearchivo"].map(build_path)

    df["ext"] = (
        df["FileName"]
        .astype(str)
        .str.rsplit(".", n=1).str[-1]
        .str.lower()
        .where(df["FileName"].astype(str).str.contains(r"\.", regex=True), "")
    )

    df["FechaNorm"] = df["Fecha"].map(normalizar_fecha_yyyy_mm_dd)


    df_pictures = df[df["ext"].isin(IMAGE_EXTS)].copy()
    df_files = df[~df["ext"].isin(IMAGE_EXTS)].copy()

    counters["rows_files_csv"] = len(df_files)
    counters["rows_pictures_csv"] = len(df_pictures)

    files_df = pd.DataFrame({
        "FileID": range(1, len(df_files) + 1),
        "FileClientID": df_files["IDpaciente"].astype(str),
        "FileName": df_files["FileName"].map(flowww_safe).str.slice(0, 255),
        "FileClassID": df_files["FileName"].map(get_file_class_id).astype(str),
        "FilePath": df_files["FlowwwPath"].map(flowww_safe).str.slice(0, 255),
        "FileDate": df_files["FechaNorm"],
        "FileComments": "",
    })

    counters["total_files_csv"] = len(files_df)

    pictures_df = pd.DataFrame({
        "PictureID": range(1, len(df_pictures) + 1),
        "PictureClientID": df_pictures["IDpaciente"].astype(str),
        "PictureComments": "",
        "PictureDate": df_pictures["FechaNorm"],
        "PicturePath": df_pictures["FlowwwPath"].map(flowww_safe).str.slice(0, 255),
    })

    counters["total_pictures_csv"] = len(pictures_df)

    if not DRY_RUN:
        files_df.to_csv(
            FILES_OUT,
            sep=CSV_SEP,
            encoding=CSV_ENCODING,
            index=False,
            na_rep="\\N",
            lineterminator="\n",
        )
        pictures_df.to_csv(
            PICTURES_OUT,
            sep=CSV_SEP,
            encoding=CSV_ENCODING,
            index=False,
            na_rep="\\N",
            lineterminator="\n",
        )

    logger.info(f"\nPATH_PREFIX usado: '{PATH_PREFIX}'  PATH_JOIN: '{PATH_JOIN}'")
    logger.info("\n=== CONTADORES ===")
    for k in sorted(counters.keys()):
        logger.info(f"{k}: {counters[k]}")

    logger.info("\n✅ Proceso finalizado" + (" (DRY_RUN)" if DRY_RUN else ""))

if __name__ == "__main__":
    main()
